# 🧼 Advanced Data Cleaning & Integrity
This notebook provides a modular, interactive workflow for cleaning the credit risk dataset, focused on integrity and readying data for feature engineering.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os

raw_path = '../data/credit_risk_dataset.csv'
df = pd.read_csv(raw_path)
print(f'Initial dataset shape: {df.shape}')
df.head()

## 1. Outlier Detection and Removal
Suspicious values identified in Age (> 90) and Employment Length (> 60).

In [ ]:
# Filter outliers
initial_count = len(df)
df = df[df['person_age'] <= 90]
df = df[df['person_emp_length'] <= 60]

# Visualization of removal impact
fig = px.bar(x=['Initial', 'After Outlier Removal'], y=[initial_count, len(df)],
             title='Dataset Size Impact: Outlier Removal', labels={'x': 'Stage', 'y': 'Row Count'})
fig.show()
print(f'Rows removed: {initial_count - len(df)}')

## 2. Duplicate Detection & Removal
Checking for identical records to avoid data leakage.

In [ ]:
# Identify and remove duplicates
dup_count = df.duplicated().sum()
df = df.drop_duplicates()

print(f'Found and removed {dup_count} duplicates.')
print(f'Current dataset shape: {df.shape}')

## 3. Logical Consistency Checks
Ensuring 'impossible' values (like Employment Length > Age) are handled.

In [ ]:
# Check: Emp Length cannot exceed Age
inconsistent_mask = df['person_emp_length'] > df['person_age']
inconsistent_count = inconsistent_mask.sum()

if inconsistent_count > 0:
    df = df[~inconsistent_mask]
    print(f'Removed {inconsistent_count} records with inconsistent Age/Employment data.')
else:
    print('All logical consistency checks passed (Age vs Employment).')

## 4. Handling Missing Values
Imputing values for `person_emp_length` and `loan_int_rate` based on EDA insights.

In [ ]:
# Impute Missing Values
# 1. Fill missing employment length with 0
df['person_emp_length'] = df['person_emp_length'].fillna(0)

# 2. Fill missing interest rates with median
median_int_rate = df['loan_int_rate'].median()
df['loan_int_rate'] = df['loan_int_rate'].fillna(median_int_rate)

# Visualizing Final Null Counts
null_counts = df.isnull().sum()
fig = px.bar(x=null_counts.index, y=null_counts.values, title='Missing Values per Feature (Final)')
fig.show()

## 5. Save Cleaned Data
Generating the final CSV for modeling.

In [ ]:
output_path = '../data/processed/credit_risk_cleaned.csv'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df.to_csv(output_path, index=False)
print(f'Final cleaned data saved to: {output_path}')
print(f'Final Shape: {df.shape}')